# Visualização da Tabela


In [16]:
# Ativa o recarregamento para não dar erro de cache
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../src"))
from queries import buscar_vendas_gerais, buscar_custos_operacionais

df_vendas = buscar_vendas_gerais()
df_custos = buscar_custos_operacionais()

c:\Users\igor.de.paula\dashInsights\src\queries.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
c:\Users\igor.de.paula\dashInsights\src\queries.py:57: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [ ]:
import config_regras as regras

# 1. Criamos a coluna 'mes_referencia' no DF de vendas (Formato YYYY-MM)
df_vendas['mes_referencia'] = df_vendas['data_venda'].dt.to_period('M').astype(str)

# 2. Mapeamos o CAC dinamicamente baseado no canal de aquisição
df_vendas['custo_cac'] = df_vendas['canal_aquisicao'].map(regras.CAC_POR_CANAL)

# 3. Faturamento Bruto Comercial (Apenas quem não está inadimplente)
faturamento_bruto = df_vendas[df_vendas['inadimplente'] == False]['valor_pago'].sum()

# =========================================================================
# LENDO AS REGRAS FISCAIS DO ARQUIVO DE CONFIGURAÇÃO CENTRALIZADO
# =========================================================================
# Impostos recalculados dinamicamente pela constante do config_regras
impostos_deducoes = faturamento_bruto * regras.ALIQUOTA_IMPOSTO

# Faturamento Líquido Real
faturamento_liquido = faturamento_bruto - impostos_deducoes

# Custo de Entrega (COGS) recalculado pela constante
custo_cogs = faturamento_bruto * regras.TAXA_COGS

# 4. Investimento Total em Marketing (Soma de todos os CACs)
total_investimento_marketing = df_vendas['custo_cac'].sum()

# 5. Custo Operacional Fixo de Estrutura (OpEx vindo do banco de dados)
custo_opex_total = df_custos['valor_custo'].sum()

# 6. Cálculo do Lucro Líquido Real Corporativo
lucro_real = faturamento_liquido - (custo_cogs + custo_opex_total + total_investimento_marketing)

# --- EXIBIÇÃO DO DRE OFICIAL ---
print("📊 ======================================================= 📊")
print("💵      DEMONSTRATIVO DE RESULTADOS (DRE PARAMETRIZADA)    💵")
print("📊 ======================================================= 📊\n")
print(f"💰 (+) FATURAMENTO BRUTO:          R$ {faturamento_bruto:,.2f}")
print(f"💸 (-) Impostos e Deduções ({regras.ALIQUOTA_IMPOSTO*100:.0f}%): R$ {impostos_deducoes:,.2f}")
print(f"🌍 (=) FATURAMENTO LÍQUIDO:        R$ {faturamento_liquido:,.2f}\n")
print(f"📦 (-) Custo de Entrega (COGS {regras.TAXA_COGS*100:.0f}%):R$ {custo_cogs:,.2f}")
print(f"📢 (-) Investimento Marketing(CAC):R$ {total_investimento_marketing:,.2f}")
print(f"📉 (-) Custo Fixo de OpEx (Banco): R$ {custo_opex_total:,.2f}\n")
print(f"🚀 (=) LUCRO LÍQUIDO REAL:         R$ {lucro_real:,.2f}")

margem_lucro = (lucro_real / faturamento_bruto) * 100 if faturamento_bruto > 0 else 0
print(f"📊 (%) Margem Líquida Real:        {margem_lucro:.2f}%\n")
print("📊 ======================================================= 📊")

NameError: name 'impuestos_deducoes' is not defined